# Data Cleaning and Validation

In [1]:
%pip install -q pandas

Note: you may need to restart the kernel to use updated packages.


## Load and inspect the raw data

In [2]:
import pandas as pd

orders = pd.read_csv('orders_raw.csv')
products = pd.read_csv('products.csv')
print(orders.shape, products.shape)

(508, 11) (31, 5)


In [3]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 508 entries, 0 to 507
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       508 non-null    int64  
 1   order_date     508 non-null    str    
 2   customer_name  508 non-null    str    
 3   city           508 non-null    str    
 4   category       508 non-null    str    
 5   product_id     508 non-null    int64  
 6   quantity       508 non-null    int64  
 7   amount_inr     498 non-null    float64
 8   payment_mode   508 non-null    str    
 9   status         508 non-null    str    
 10  rating         439 non-null    float64
dtypes: float64(2), int64(3), str(6)
memory usage: 43.8 KB


In [4]:
orders.describe()

,order_id,product_id,quantity,amount_inr,rating
count,508.000000,508.000000,508.000000,498.000000,439.000000
mean,250.204724,15.476378,3.045276,257.530120,3.082005
std,144.594139,8.747467,1.364497,559.424136,1.364126
min,1.000000,1.000000,1.000000,20.000000,1.000000
25%,125.750000,8.000000,2.000000,90.000000,2.000000
50%,249.500000,15.000000,3.000000,168.000000,3.000000
75%,376.250000,23.000000,4.000000,283.750000,4.000000
max,500.000000,30.000000,5.000000,7600.000000,5.000000


In [5]:
orders['status'].value_counts()

status
Delivered    439
Cancelled     43
Pending       26
Name: count, dtype: int64

The inspection above shows four data quality issues:

1. The raw file has more rows than expected, 508 instead of 500, because 8 order_id values are duplicated.
2. The city and category columns have mixed casing and stray whitespace, for example ' Bengaluru ', 'BENGALURU', and 'bengaluru' all referring to the same city.
3. amount_inr has missing values, 10 rows have no value recorded.
4. amount_inr has suspiciously large values, the median is around 168 but the maximum reaches 7600, far above the 75th percentile.

## Remove duplicate rows

In [6]:
rows_before = len(orders)
orders = orders.drop_duplicates(subset='order_id', keep='first').reset_index(drop=True)
rows_after = len(orders)
rows_removed = rows_before - rows_after
print('rows before:', rows_before)
print('rows after:', rows_after)
print('rows removed:', rows_removed)

rows before: 508
rows after: 500
rows removed: 8


## Fix casing and whitespace

In [7]:
orders['city'] = orders['city'].str.strip().str.title()
orders['category'] = orders['category'].str.strip().str.title()

city_values = sorted(orders['city'].unique())
category_values = sorted(orders['category'].unique())

print('distinct city values:', city_values)
print('distinct category values:', category_values)
print('city count:', len(city_values))
print('category count:', len(category_values))

distinct city values: ['Bengaluru', 'Hyderabad', 'Mumbai', 'Pune']
distinct category values: ['Bakery', 'Dairy & Eggs', 'Fruits & Vegetables', 'Household Essentials', 'Personal Care', 'Snacks & Beverages']
city count: 4
category count: 6
